# Retrieval-Augmented Generation (RAG)

In this notebook we will build a functional RAG system step by step, from loading documents to evaluating the complete pipeline.

## Setup

In [ ]:
# Install dependencies
!pip install -q langchain langchain-community langchain-openai
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q chromadb
!pip install -q datasets
!pip install -q tiktoken

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import textwrap
import numpy as np
import pandas as pd
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

### API configuration

For this notebook, we will use an LLM provided by OpenAI.

In [ ]:
import os
import getpass
TOKEN = getpass.getpass("Introduce your token: ")
os.environ["OPENAI_API_KEY"] = TOKEN

## Data Loading and Exploration

We will use a small corpus of NLP-related articles as our knowledge base. It is small enough to make all mechanisms visible, but realistic enough to be meaningful.

In a real project, this is where you would load your PDFs, Word documents, web pages, etc.

In [ ]:
# Topics related to NLP
nlp_topics = [
    "Natural language processing",
    "Word embedding",
    "Transformer architecture",
    "BERT model",
    "GPT model",
    "Recurrent neural network",
    "Attention mechanism",
    "Named-entity recognition",
    "Text classification",
    "Sentiment analysis",
]

print(f"Selected topics: {len(nlp_topics)}")

In [ ]:
corpus = {
    "NLP Overview": '''
    Natural language processing (NLP) is a subfield of linguistics, computer science,
    and artificial intelligence concerned with the interactions between computers and human language.
    NLP tasks include text classification, named entity recognition, machine translation,
    question answering, and text summarization. Modern NLP relies heavily on deep learning
    and large language models trained on massive text corpora.
    Key challenges in NLP include ambiguity, context dependence, and the diversity of human languages.
    ''',

    "Word Embeddings": '''
    Word embeddings are dense vector representations of words in a continuous vector space.
    Unlike one-hot encodings, embeddings capture semantic relationships between words.
    Word2Vec, introduced by Mikolov et al. in 2013, uses either CBOW or Skip-gram architectures
    to learn embeddings from large text corpora. GloVe (Global Vectors) combines global
    matrix factorization with local context window methods. FastText extends Word2Vec
    by representing words as bags of character n-grams, allowing it to handle out-of-vocabulary words.
    Embeddings encode analogies: king - man + woman ~= queen.
    ''',

    "Transformer Architecture": '''
    The Transformer architecture, introduced in 'Attention is All You Need' (Vaswani et al., 2017),
    revolutionized NLP by replacing recurrent networks with self-attention mechanisms.
    The encoder-decoder structure uses multi-head attention to capture relationships between all
    positions in a sequence simultaneously. Positional encodings are added to preserve sequence order.
    The Transformer enabled the development of BERT, GPT, T5, and most modern large language models.
    Self-attention computes queries, keys, and values from the input, allowing each token to attend
    to all other tokens regardless of distance.
    ''',

    "BERT": '''
    BERT (Bidirectional Encoder Representations from Transformers) was introduced by Devlin et al.
    at Google in 2018. It is pre-trained using masked language modeling (MLM) and next sentence
    prediction (NSP) on large text corpora. BERT's bidirectional nature allows it to capture context
    from both left and right of each token, unlike GPT which is unidirectional.
    Fine-tuning BERT on downstream tasks such as classification, NER, and QA achieves
    state-of-the-art results with relatively little task-specific data.
    BERT-base has 12 transformer layers, 768 hidden units, and 110M parameters.
    ''',

    "GPT and Language Models": '''
    GPT (Generative Pre-trained Transformer) models are autoregressive language models that predict
    the next token given all previous tokens. GPT-3 (Brown et al., 2020) demonstrated that large
    language models can perform few-shot learning, adapting to new tasks with just a few examples
    in the prompt. ChatGPT and GPT-4 are instruction-tuned versions that use Reinforcement Learning
    from Human Feedback (RLHF) to align model outputs with human preferences.
    The key insight is that scale (more data, more parameters) leads to emergent capabilities.
    ''',

    "Named Entity Recognition": '''
    Named Entity Recognition (NER) is the task of identifying and classifying named entities
    in text into predefined categories such as person names, organizations, locations,
    medical codes, time expressions, quantities, and monetary values.
    Traditional NER used rule-based systems and CRFs (Conditional Random Fields).
    Modern NER uses BERT-based models fine-tuned on annotated corpora like CoNLL-2003.
    BIO tagging (Beginning, Inside, Outside) is the standard annotation scheme.
    Evaluation uses entity-level F1 score. SpaCy and Hugging Face Transformers provide
    pre-trained NER models for multiple languages.
    ''',

    "Text Classification": '''
    Text classification assigns predefined categories to text documents.
    Applications include sentiment analysis, spam detection, topic labeling, and intent recognition.
    Classical approaches use TF-IDF features with logistic regression or SVMs.
    Deep learning approaches use CNNs, LSTMs, or fine-tuned BERT models.
    Sentiment analysis classifies text as positive, negative, or neutral.
    Multi-label classification assigns multiple categories to a single document.
    Zero-shot classification uses LLMs to classify without task-specific training data.
    ''',

    "Retrieval and Search": '''
    Information retrieval systems find relevant documents in response to a user query.
    TF-IDF and BM25 are classic sparse retrieval methods based on term frequency statistics.
    Dense retrieval uses neural embeddings to represent queries and documents in a shared vector space.
    Bi-encoders encode query and document independently; cross-encoders process them jointly for reranking.
    Approximate nearest neighbor (ANN) algorithms like FAISS enable efficient search over millions of vectors.
    BEIR is a benchmark for evaluating retrieval systems across diverse domains.
    Hybrid retrieval combines sparse and dense methods for better coverage.
    ''',

    "Evaluation Metrics in NLP": '''
    NLP systems are evaluated using task-specific metrics.
    For classification: accuracy, precision, recall, F1 score, and ROC-AUC.
    For generation: BLEU measures n-gram overlap with reference translations.
    ROUGE measures recall-oriented overlap, commonly used for summarization.
    BERTScore uses contextual embeddings to measure semantic similarity.
    For retrieval: Precision@k, Recall@k, MRR (Mean Reciprocal Rank), and NDCG.
    Human evaluation remains important for generation tasks where automatic metrics fall short.
    Perplexity measures how well a language model predicts a held-out corpus.
    ''',

    "Attention Mechanism": '''
    The attention mechanism allows neural networks to focus on relevant parts of the input
    when producing an output. Introduced by Bahdanau et al. (2015) for machine translation,
    attention computes a weighted sum of encoder hidden states based on their relevance to
    the current decoder state. Self-attention, used in Transformers, relates different positions
    of a single sequence to compute a representation. Multi-head attention runs multiple attention
    operations in parallel, capturing different types of relationships. Scaled dot-product attention
    divides the dot product by the square root of the key dimension to prevent vanishing gradients.
    ''',
}

print(f"Corpus loaded: {len(corpus)} documents")
for title, text in corpus.items():
    print(f"  · {title}: {len(text.split())} words")

### Length Histogram

In [ ]:
import matplotlib.pyplot as plt

lengths = [len(text) for text in corpus.values()]

plt.figure(figsize=(10, 6))
plt.hist(lengths, bins=50, edgecolor='black')
plt.xlabel('Text Length (characters)')
plt.ylabel('Number of Articles')
plt.title('Distribution of Article Text Lengths')
plt.tight_layout()
plt.show()

## Chunking: strategies and comparison

Chunking is one of the most important decisions in a RAG system. We will implement and compare three strategies.

**Question to think about:** What happens if chunks are too small? What if they are too large?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_core.documents import Document

# Convert the corpus to LangChain Document objects
documents = [
    Document(page_content=text, metadata={"title": title})
    for title, text in corpus.items()
]

print(f"Documents ready: {len(documents)}")

In [ ]:
# Strategy 1: Fixed size with overlap
splitter_fixed = RecursiveCharacterTextSplitter(
    chunk_size=300,       # characters per chunk
    chunk_overlap=50,     # overlap between consecutive chunks
    length_function=len,
)

chunks_small = splitter_fixed.split_documents(documents)

print(f"Strategy 1: Fixed size (300 chars, 50 overlap)")
print(f"  Total chunks: {len(chunks_small)}")
print(f"  Average size: {np.mean([len(c.page_content) for c in chunks_small]):.0f} chars")
print()
print("Example chunk:")
print("-" * 60)
print(chunks_small[5].page_content)
print(f"[Source: {chunks_small[5].metadata['title']}]")

In [ ]:
# Strategy 2: Larger chunks
splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    length_function=len,
)

chunks_large = splitter_large.split_documents(documents)

print(f"Strategy 2: Large chunks (600 chars, 100 overlap)")
print(f"  Total chunks: {len(chunks_large)}")
print(f"  Average size: {np.mean([len(c.page_content) for c in chunks_large]):.0f} chars")

In [ ]:
# Strategy 3: Sentence-aware (simple semantic approximation)
splitter_sentence = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=[". ", "\n\n", "\n", " ", ""],  # prioritize sentence boundaries
    length_function=len,
)

chunks_sentence = splitter_sentence.split_documents(documents)

print(f"Strategy 3: Sentence-aware (400 chars, semantic separators)")
print(f"  Total chunks: {len(chunks_sentence)}")
print(f"  Average size: {np.mean([len(c.page_content) for c in chunks_sentence]):.0f} chars")

In [ ]:
# Visual comparison

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

strategies = [
    ("Small (300)", chunks_small),
    ("Large (600)", chunks_large),
    ("Sentence-aware (400)", chunks_sentence),
]

colors = ["red", "green", "blue"]

for ax, (name, chunks), color in zip(axes, strategies, colors):
    sizes = [len(c.page_content) for c in chunks]
    ax.hist(sizes, bins=20, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(f"{name}\n({len(chunks)} chunks)", fontweight="bold")
    ax.set_xlabel("Size (chars)")
    ax.axvline(np.mean(sizes), color="red", linestyle="--", linewidth=1.5,
               label=f"Mean: {np.mean(sizes):.0f}")
    ax.legend(fontsize=9)

axes[0].set_ylabel("Frequency")
plt.suptitle("Chunk size distribution by strategy", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### Questions

1. Which strategy produces the most uniform chunks? Is that always desirable?
2. Find a chunk from the fixed strategy that cuts in the middle of an idea. How would that affect retrieval?
3. What would be the effect of increasing the overlap to 50%?

## Embeddings and indexing with FAISS

We now convert chunks into vectors and store them in a vectorstore. We use `sentence-transformers` for embeddings and FAISS for efficient similarity search.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Embedding model: all-MiniLM-L6-v2
# - Lightweight (80 MB), fast, good quality for English
# - Produces 384-dimensional vectors

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Quick sanity check
test_vec = embedding_model.embed_query("What is BERT?")
print(f"Embedding dimension: {len(test_vec)}")
print(f"First values: {test_vec[:5]}")

In [ ]:
# Index the chunks with FAISS
# We use the fixed-size strategy (chunks_fixed) as the baseline

print("Indexing chunks... (may take a few seconds)")

vectorstore = FAISS.from_documents(
    documents=chunks_large,
    embedding=embedding_model,
)

print(f"Vectorstore created")
print(f"  Indexed chunks: {len(chunks_large)}")
print(f"  Vector dimension: {vectorstore.index.d}")

In [ ]:
# Save the index so we do not need to rebuild it
vectorstore.save_local("faiss_index")
print("Index saved to ./faiss_index/")

# To reload later:
# vectorstore = FAISS.load_local("faiss_index", embedding_model,
#                                allow_dangerous_deserialization=True)

### Visualizing the embedding space

We reduce dimensionality with PCA to visualize how chunks are distributed in the vector space.

In [ ]:
from sklearn.decomposition import PCA

# Extract all vectors from the FAISS index
index = vectorstore.index
n_vectors = index.ntotal
dimension = index.d

all_vectors = np.zeros((n_vectors, dimension), dtype=np.float32)
index.reconstruct_n(0, n_vectors, all_vectors)

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
vectors_2d = pca.fit_transform(all_vectors)

# Colour by source document
doc_titles = [c.metadata["title"] for c in chunks_large]
unique_titles = list(set(doc_titles))
color_map = {t: plt.cm.tab10(i / len(unique_titles)) for i, t in enumerate(unique_titles)}
colors_plot = [color_map[t] for t in doc_titles]

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(vectors_2d[:, 0], vectors_2d[:, 1],
           c=colors_plot, alpha=0.7, s=50, edgecolors="white", linewidth=0.3)

patches = [mpatches.Patch(color=color_map[t], label=t) for t in unique_titles]
ax.legend(handles=patches, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)
ax.set_title("Chunk embeddings (PCA 2D)", fontsize=13, fontweight="bold")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.tight_layout()
plt.show()

print(f"Variance explained by 2 components: {sum(pca.explained_variance_ratio_)*100:.1f}%")

## Retrieval: similarity search

With the index built, we can now retrieve the most relevant chunks for any query.


In [ ]:
# Basic retrieval function with result analysis

def retrieve(query: str, k: int = 3, show_scores: bool = True):
    """
    Retrieves the k most relevant chunks for the query.
    Displays similarity scores and content for each chunk.
    """
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"Query: '{query}'")
    print(f"Top-{k} retrieved chunks:")
    print("=" * 70)

    for i, (doc, score) in enumerate(results, 1):
        # FAISS returns L2 distance; convert to approximate similarity
        similarity = 1 / (1 + score)
        print(f"\n[{i}] Source: {doc.metadata['title']}")
        print(f"    Similarity: {similarity:.4f}  (L2 distance: {score:.4f})")
        print(f"    Content: {doc.page_content[:200]}...")

    return results

# Test with a query
results = retrieve("How does the attention mechanism work in transformers?", k=3)

In [ ]:
# More retrieval examples

queries = [
    "What is BERT and how is it pre-trained?",
    "How are word embeddings trained?",
    "What metrics are used to evaluate NER systems?",
]

for q in queries:
    print("\n" + "-"*70)
    retrieve(q, k=2)

### Experiment: comparing chunking strategies in retrieval

Compare retrieval results using small vs. large chunks for the same query.


In [ ]:
# Build two vectorstores with different strategies for comparison

vs_small = FAISS.from_documents(chunks_small, embedding_model)    # 300 chars
vs_large = FAISS.from_documents(chunks_large, embedding_model)    # 600 chars

query_comparison = "How does self-attention capture relationships between words?"

print("SMALL CHUNKS (300 chars):")
print("-" * 50)
for doc, score in vs_small.similarity_search_with_score(query_comparison, k=2):
    print(f"[{doc.metadata['title']}]")
    print(f"{doc.page_content[:250]}\n")

print("\nLARGE CHUNKS (600 chars):")
print("-" * 50)
for doc, score in vs_large.similarity_search_with_score(query_comparison, k=2):
    print(f"[{doc.metadata['title']}]")
    print(f"{doc.page_content[:350]}\n")

## Augmented generation: the complete RAG pipeline

We now connect retrieval to an LLM to generate answers grounded in the retrieved context.

As an LLM, we will use the OpenAI API. It requires a key that has a cost and will be available during the session only.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Augmented prompt template
RAG_PROMPT_TEMPLATE = """You are an expert assistant in Natural Language Processing.
Answer the question using ONLY the information provided in the context below.
If the answer is not in the context, say "I don't have enough information to answer this question."
Always mention which source(s) you used.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:"""

prompt = PromptTemplate(
    template=RAG_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

print("Prompt template created")
print("\nExample prompt with placeholder context:")
print("-" * 60)
print(prompt.format(
    context="[Source 1] BERT uses masked language modeling...\n[Source 2] BERT was introduced in 2018...",
    question="When was BERT introduced?"
))

In [ ]:
# Prepare OpenAI model

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=256)

In [ ]:
# Context formatting function

def format_context(docs):
    """Formats retrieved chunks for inclusion in the prompt."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(f"[Source {i}: {doc.metadata['title']}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# Complete RAG chain
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {
        "context": retriever | format_context,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built")

In [ ]:
test = llm.invoke("Answer the question: What is BERT? Answer:")
print(repr(test))

In [ ]:
# Test the complete system

def ask_rag(question: str, verbose: bool = True):
    """
    Asks a question to the RAG system and shows the full process.
    """
    if verbose:
        retrieved_docs = retriever.invoke(question)
        print(f"Question: {question}")
        print(f"\nRetrieved chunks:")
        for i, doc in enumerate(retrieved_docs, 1):
            print(f"  [{i}] {doc.metadata['title']}: {doc.page_content[:100]}...")
        print()

    answer = rag_chain.invoke(question)
    print(f"Answer:\n{answer}")
    print("─" * 70)
    return answer

# Examples
ask_rag("What is the difference between BERT and GPT?")

In [ ]:
ask_rag("How does TF-IDF differ from dense retrieval?")

In [ ]:
# Out-of-domain question (should admit it does not know)
ask_rag("What is the capital of France?")

### Questions for reflection

1. Does the system answer correctly when the question is outside the corpus? How could you improve this?
2. Compare the RAG answer with what the LLM would say without context. How do they differ?
3. Change `k` in the retriever (from 3 to 1, or to 6) and observe how the answers change.
